# OMNI-X21 · The forecaster trained to forecast: versions B, A, C (Colab GPU)

D-65 (committed before this notebook first ran). Three versions, 3 seeds each, all with `minus_first`, labels
Event or Normal, Fix 1 (each event second's class error weighted by `exp(-max(0, τ - τ0) / s)`):

| Version | Datasets | Fix 2 (ranking inside the window) | Folder |
|---|---|---|---|
| **B** | CCD + UCF-Crime | ✓ | `models/pfm/v5-b/seed*/` |
| **A** | CCD + UCF-Crime | | `models/pfm/v5-a/seed*/` |
| **C** | UCF-Crime only | ✓ | `models/pfm/v5-c/seed*/` |

Order **B, A, C** (the served model comes from A or B). AdamW, lr 3e-4, weight decay 0.01, batch 256, gradient clip
1.0, up to 120 epochs, patience 30, `α_d = 1 / var(τ)` of each dataset's train split.

**The epoch (D-65):** after every epoch all eight measures on validation go into the history. An epoch can become
`best.pt` only if **UCF-Crime's flat-line check is ≤ 0.60**; among those, the highest **UCF-Crime within-window
AUC** wins (the first on a tie), and the patience counts from it. A seed with no passing epoch has no `best.pt`.

1. **Setup:** clone, Drive
2. **The vectors:** as OMNI-905
3. **Training:** resumable: if the session ends, Run all again and every seed continues where it stopped.
   `RUN_ONLY` limits it to some versions (e.g. `("C",)` on Sunday)
4. **Per version:** the best epoch of each seed on validation, `results/x21_v5{version}.json`

Choosing the served model and held-out are OMNI-X22, not here. Held-out data is not touched.
Runtime → Change runtime type → **GPU**.

In [ ]:
# 1 · Setup
BRANCH = "slice-09-10-forecast"
REPO = "https://github.com/OmniSight-team/omnisight.git"

import os, subprocess, sys
if not os.path.isdir("/content/omni-sight"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/omni-sight"], check=True)
else:
    subprocess.run(["git", "-C", "/content/omni-sight", "pull", "-q"], check=True)
COMMIT = subprocess.run(["git", "-C", "/content/omni-sight", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)
sys.path.insert(0, "/content/omni-sight/scripts")

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
DEVICE, GPU = "cuda", torch.cuda.get_device_name(0)
print(torch.__version__, GPU)
REPO_DIR = "/content/omni-sight"
DRIVE_DIR = "/content/drive/MyDrive/omnisight"
LOCAL_DIR = "/content/pfm"

In [ ]:
# 2 · Both domains' InternVideo2 vectors, copied from Drive once, and the samples
import json, os, shutil, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pfm_samples, train_pfm
from pfm_model import DISTANCE_WEIGHTS, RANK_GAPS, Config

REPO_DIR, DRIVE, LOCAL = Path(REPO_DIR), Path(DRIVE_DIR), Path(LOCAL_DIR)
DOMAINS = ("ccd", "ucf_crime")
SAMPLE_FILES = {d: REPO_DIR / "datasets" / "samples" / f"{d}.csv" for d in DOMAINS}
SAMPLES = {d: pfm_samples.read_samples(SAMPLE_FILES[d]) for d in DOMAINS}

def copy_folder(domain):
    keys = sorted({s["clip_key"] for s in SAMPLES[domain]})
    source, target = DRIVE / "embeddings" / "pfm" / domain / "internvideo2", LOCAL / domain
    def one(key):
        dst = target / f"{key}.npz"
        if not dst.is_file():  # copied under a temporary name, renamed when complete: a cut copy leaves no broken file
            dst.parent.mkdir(parents=True, exist_ok=True)
            part = dst.with_name(dst.name + ".part")
            shutil.copyfile(source / f"{key}.npz", part)
            os.replace(part, dst)
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(one, keys))
    return target

TRAIN, VAL = {}, {}
for d in DOMAINS:
    start = time.time()
    store = pfm_samples.Store(SAMPLES[d], copy_folder(d))
    TRAIN[d] = train_pfm.Data.split(store, SAMPLES[d], "train", d).for_labels("event")  # D-62
    VAL[d] = train_pfm.Data.split(store, SAMPLES[d], "validation", d).for_labels("event")
    print(f"{d}: {len(store.vectors):,} vectors, train {len(TRAIN[d]):,} samples, validation {len(VAL[d]):,} "
          f"({time.time() - start:.0f} s)")

In [ ]:
# 3 · Training: versions B, A, C (D-65), 3 seeds each, resumable
VERSIONS = {"B": {"domains": DOMAINS, "rank": True},
            "A": {"domains": DOMAINS, "rank": False},
            "C": {"domains": ("ucf_crime",), "rank": True}}
ORDER = ("B", "A", "C")
RUN_ONLY = ORDER  # e.g. ("C",) to train only C
SEEDS = (0, 1, 2)
EPOCHS, PATIENCE = 120, 30
ALPHAS = {d: float(f"{1.0 / np.nanvar(TRAIN[d].tau):.3g}") for d in DOMAINS}  # D-59: 1 / var(τ) of train positives
print("alphas", ALPHAS)

def config_of(version):
    v = VERSIONS[version]
    domains = list(v["domains"])
    return Config(input_dim=512, domains=domains, alphas={d: ALPHAS[d] for d in domains}, input="minus_first",
                  labels="event", epochs=EPOCHS, distance_weights={d: DISTANCE_WEIGHTS[d] for d in domains},
                  rank_gaps={d: RANK_GAPS[d] for d in domains} if v["rank"] else {})

def folder_of(version, seed):
    return DRIVE / "models" / "pfm" / f"v5-{version.lower()}" / f"seed{seed}"

HISTORY = {}
for version in ORDER:
    if version not in RUN_ONLY:
        continue
    config, domains = config_of(version), VERSIONS[version]["domains"]
    val = {d: VAL[d] for d in domains}
    for seed in SEEDS:
        print(f"version {version}, seed {seed}")
        _, HISTORY[version, seed] = train_pfm.train(
            config, {d: TRAIN[d] for d in domains}, folder_of(version, seed), seed=seed, device=DEVICE,
            validate=lambda model: train_pfm.validation(model, val, DEVICE, choose_by="ucf_crime"), patience=PATIENCE)

In [ ]:
# 4 · Per version: each seed's best epoch on validation (from the history: best.pt is that epoch), the record
def f(v, spec=".4f"):
    return "-" if v is None else format(v, spec)

KEYS = [("within_window_auc", ".4f"), ("flat_line", ".4f"), ("mtta_s", ".2f"), ("detection_rate", ".3f"),
        ("false_alarm_windows", ".3f"), ("false_alarms_per_hour", ".1f"), ("ap", ".4f"), ("per_window_auc", ".4f"),
        ("pooled_auc", ".4f"), ("theta", ".4f")]

for version in ORDER:
    if not all((version, s) in HISTORY for s in SEEDS):
        continue
    config, domains = config_of(version), VERSIONS[version]["domains"]
    best = {}
    for s in SEEDS:
        passing = [h for h in HISTORY[version, s] if h["validation"]["passes"]]
        best[s] = max(passing, key=lambda h: (h["validation"]["score"], -h["epoch"])) if passing else None
        if best[s] is not None:  # best.pt holds that epoch
            assert train_pfm.load(folder_of(version, s) / "best.pt", "cpu")[1]["epoch"] == best[s]["epoch"]
        else:
            assert not (folder_of(version, s) / "best.pt").exists()
    print(f"\nversion {version}")
    for s in SEEDS:
        h = HISTORY[version, s]
        print(f"  seed {s}: {len(h)} epochs trained, {sum(x['validation']['passes'] for x in h)} passed the flat-line "
              f"check, best " + (f"epoch {best[s]['epoch']}" if best[s] else "none (failed: no passing epoch)"))
    for d in domains:
        print(f"  {d} (validation, each seed's best epoch)")
        print("    seed  " + "  ".join(k for k, _ in KEYS) + "  windows (within / flat)")
        for s in SEEDS:
            if best[s] is None:
                print(f"    {s}     failed")
                continue
            m = best[s]["validation"][d]
            print(f"    {s}     " + "  ".join(f(m[k], spec) for k, spec in KEYS)
                  + f"  {m['within_window_windows']}+{m['within_window_skipped']} skipped / "
                    f"{m['flat_line_windows']}+{m['flat_line_skipped']} skipped")
    out = DRIVE / "results" / f"x21_v5{version.lower()}.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps({
        "version": version, "commit": COMMIT, "gpu": GPU, "torch": torch.__version__, "config": config.to_dict(),
        "seeds": SEEDS, "patience": PATIENCE,
        "best_per_seed": {s: None if b is None else {"epoch": b["epoch"], "score": b["validation"]["score"],
                                                      "validation": b["validation"]} for s, b in best.items()},
        "history": {s: HISTORY[version, s] for s in SEEDS},
        "train_seconds": {s: sum(h["seconds"] for h in HISTORY[version, s]) for s in SEEDS}},
        indent=1, default=float))
    print(" ", out)